In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [15]:
import torch
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler, DataCollatorWithPadding, TrainingArguments, Trainer, EarlyStoppingCallback
from torch.optim import AdamW
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import evaluate

## LLama 2

In [16]:
model_llama = "meta-llama/Llama-2-7b-chat-hf"

In [17]:
tokenizer = AutoTokenizer.from_pretrained(model_llama)
model = AutoModelForCausalLM.from_pretrained(
    model_llama,
    torch_dtype=torch.float16,
    device_map="auto",
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [18]:
# Use model explicitly for inference clearly
prompt = "The highest mountain in the world is"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_length=256)

response = tokenizer.decode(output.squeeze(), skip_special_tokens=True)
print(response)

The highest mountain in the world is Mount Everest, located in the Himalayas on the border between Nepal and Tibet, China. It stands at an elevation of 8,848 meters (29,029 feet) above sea level.

Mount Everest is also known as Chomolungma or Sagarmatha, which means "Mother Goddess of the Universe" in the local Sherpa language. It is considered one of the most challenging and dangerous mountains to climb, due to its extreme altitude and harsh weather conditions.

Despite the risks, many climbers attempt to reach the summit of Mount Everest every year. The first successful ascent was made by Sir Edmund Hillary and Tenzing Norgay in 1953. Since then, hundreds of climbers have reached the summit, including many famous mountaineers and adventurers.

In addition to its natural beauty and challenging climb, Mount Everest has also become a popular destination for tourists and adventure seekers. Many people visit the mountain base camp to experience the stunning scenery and to get a taste of t

## OPT

In [31]:
model_opt = "facebook/opt-350m"

In [32]:
tokenizer = AutoTokenizer.from_pretrained(model_opt)
model = AutoModelForCausalLM.from_pretrained(
    model_opt,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
# Use model explicitly for inference clearly
prompt = "What is the highest mountain in the world?"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    output = model.generate(**inputs, max_length=100)

response = tokenizer.decode(output.squeeze(), skip_special_tokens=True)
print(response)

What is the highest mountain in the world?

The highest mountain in the world is Mount Everest. It is the highest mountain in the world and is located in Nepal. It is the highest mountain in the world and is located in Nepal.

Mount Everest is the highest mountain in the world and is located in Nepal. It is the highest mountain in the world and is located in Nepal.

Mount Everest is the highest mountain in the world and is located in Nepal. It is


In [34]:
model.generate??

Signature:
model.generate(
    inputs: Optional[torch.Tensor] = None,
    generation_config: Optional[transformers.generation.configuration_utils.GenerationConfig] = None,
    logits_processor: Optional[transformers.generation.logits_process.LogitsProcessorList] = None,
    stopping_criteria: Optional[transformers.generation.stopping_criteria.StoppingCriteriaList] = None,
    prefix_allowed_tokens_fn: Optional[Callable[[int, torch.Tensor], List[int]]] = None,
    synced_gpus: Optional[bool] = None,
    assistant_model: Optional[ForwardRef('PreTrainedModel')] = None,
    streamer: Optional[ForwardRef('BaseStreamer')] = None,
    negative_prompt_ids: Optional[torch.Tensor] = None,
    negative_prompt_attention_mask: Optional[torch.Tensor] = None,
    **kwargs,
) -> Union[transformers.generation.utils.GenerateDecoderOnlyOutput, transformers.generation.utils.GenerateEncoderDecoderOutput, transformers.generation.utils.GenerateBeamDecoderOnlyOutput, transformers.generation.utils.GenerateBeam